## ChromaDB 

In [1]:
### Building Langchain with ChromaDB

In [2]:
import os



In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma


#utility
import numpy as np
from typing import List,Dict,Any


C:\Users\kanha\AppData\Local\Temp\ipykernel_19484\3392625172.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


# 1) Document Loading By creating sample data

In [4]:
## create sample data

sample_docs=[
     """
Machine Learning Fundamentals

Machine Learning (ML) is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed.
The main types of machine learning are supervised learning, unsupervised
learning, and reinforcement learning. Common algorithms include Linear
Regression, Logistic Regression, Decision Trees, Random Forests, Support
Vector Machines, and K-Means clustering. Model evaluation metrics include
accuracy, precision, recall, F1-score, and ROC-AUC.
""",
"""
Deep Learning Fundamental

Deep Learning is a subset of machine learning that uses neural networks with
multiple hidden layers. A neural network consists of input, hidden, and output
layers connected by weighted edges. Popular architectures include CNNs for
computer vision, RNNs and LSTMs for sequential data, and Transformers for
natural language processing. Training uses backpropagation and gradient descent
to minimize a loss function.
""",
 """
Large Language Model Fundamentals

Large Language Models (LLMs) are transformer-based neural networks trained on
massive text datasets. They generate text by predicting the next token in a
sequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts
include tokenization, embeddings, context windows, prompting, fine-tuning,
Retrieval-Augmented Generation (RAG), and inference.
"""
]
sample_docs

['\nMachine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning, and reinforcement learning. Common algorithms include Linear\nRegression, Logistic Regression, Decision Trees, Random Forests, Support\nVector Machines, and K-Means clustering. Model evaluation metrics include\naccuracy, precision, recall, F1-score, and ROC-AUC.\n',
 '\nDeep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision, RNNs and LSTMs for sequential data, and Transformers for\nnatural language processing. Training uses backpropagation and gradient descent\nto minimize a loss function.\n',
 '\nL

In [5]:
##save sample
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)
print(f"Sample doc created at :-> {temp_dir}")

Sample doc created at :-> C:\Users\kanha\AppData\Local\Temp\tmpbivw5s14


## 2) Document Loader

In [6]:
from langchain_community.document_loaders import DirectoryLoader
loader=DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)
documents=loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200]+"...")

Loaded 3 documents

First document preview:

Machine Learning Fundamentals

Machine Learning (ML) is a branch of artificial intelligence that enables
computers to learn patterns from data without being explicitly programmed.
The main types of m...


## 3) Document Splitting

In [7]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "])
chunks=text_splitter.split_documents(documents=documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
print(f"\nChunk example")
print(f"Content {chunks[0].page_content[:150]}....\n")
print(f"Metadata {chunks[0].metadata}")

Created 4 chunks from 3 documents.

Chunk example
Content Machine Learning Fundamentals

Machine Learning (ML) is a branch of artificial intelligence that enables
computers to learn patterns from data without....

Metadata {'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbivw5s14\\doc_0.txt'}


In [8]:
chunks

[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbivw5s14\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning, and reinforcement learning. Common algorithms include Linear\nRegression, Logistic Regression, Decision Trees, Random Forests, Support\nVector Machines, and K-Means clustering. Model evaluation metrics include\naccuracy, precision, recall,'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbivw5s14\\doc_0.txt'}, page_content='metrics include\naccuracy, precision, recall, F1-score, and ROC-AUC.'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbivw5s14\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses

### 4) Embedding Models


In [9]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

## 5) ChromaDB Time

In [10]:
## create vector store
persist_directory="./chroma_db"

##Initialisation chromadb with hugging face
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(
          model_name="sentence-transformers/all-MiniLM-L6-v2"
    ),
    persist_directory=persist_directory,
    collection_name="rag_collection",
   

)
print(f" Vector store created at {persist_directory} with {vectorstore._collection.count()} vectors")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Vector store created at ./chroma_db with 20 vectors


## Test similiarity Check

In [11]:
query="What are the types of machine learning?"
similiar_docs=vectorstore.similarity_search(query,k=4)
similiar_docs

[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning, and reinforcement learning. Common algorithms include Linear\nRegression, Logistic Regression, Decision Trees, Random Forests, Support\nVector Machines, and K-Means clustering. Model evaluation metrics include\naccuracy, precision, recall,'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsupervised\nlearning,

In [12]:
query="What is NLP"
similiar_docs=vectorstore.similarity_search(query,k=4)
similiar_docs

[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpayyjeji4\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generati

## Advanced similiaruty search

In [13]:
results=vectorstore.similarity_search_with_score(query,k=4)
results

[(Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'),
  1.446568250656128),
 (Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpayyjeji4\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetr

In [14]:
results=vectorstore.similarity_search_with_relevance_scores(query,k=4)
results

C:\Users\kanha\AppData\Local\Temp\ipykernel_19484\3467109865.py:1: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'), -0.02287821948810942), (Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpayyjeji4\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include G

[(Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nRetrieval-Augmented Generation (RAG), and inference.'),
  -0.02287821948810942),
 (Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpayyjeji4\\doc_2.txt'}, page_content='Large Language Model Fundamentals\n\nLarge Language Models (LLMs) are transformer-based neural networks trained on\nmassive text datasets. They generate text by predicting the next token in a\nsequence. Popular LLMs include GPT, Llama, Gemini, and Mistral. Key concepts\ninclude tokenization, embeddings, context windows, prompting, fine-tuning,\nR

### Initialize LLM ,RAG Chain,Promp Template,Query The RAG System

In [15]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")

In [16]:
llm_pred=llm.invoke("What is LLM")
print(llm_pred)

content='LLM stands for Large Language Model. It refers to a type of artificial intelligence (AI) model that is trained on vast amounts of text data, enabling it to understand and generate human-like language.\n\nLarge Language Models are typically trained using deep learning techniques, such as transformers, and can process vast amounts of unstructured text data. This training enables the models to learn patterns, relationships, and context within language, allowing them to:\n\n1. **Understand natural language**: LLMs can comprehend complex sentences, idioms, and nuances of human language.\n2. **Generate text**: They can produce coherent and grammatically correct text on a given topic or prompt.\n3. **Translate languages**: Some LLMs can translate languages with high accuracy.\n4. **Summarize content**: They can summarize long documents or articles into concise summaries.\n\nThe term "Large Language Model" was popularized by the release of the BERT (Bidirectional Encoder Representatio

## Modern RAG Chain

In [17]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [18]:
### Convert vector store to retriever

retriever=vectorstore.as_retriever(
    search_kwarg={"k":3}

)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002CE86CE4810>, search_kwargs={})

In [19]:
## Create prompt template
system_prompt="""You are an asisstant for question-answering tasks.
use the follwing pieceof retrieved context to answer the question
If you don't know the answer,just sat that you don't know.
Use three senetenes maximum and keep the answer concise.
Context:{context}"""
prompt=ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [20]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't know the answer,just sat that you don't know.\nUse three senetenes maximum and keep the answer concise.\nContext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [21]:
### Create a document chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't know the answer,just sat that you don't know.\nUse three senetenes maximum and keep the answer concise.\nContext:{context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2')
| StrOutputPars

In [22]:
### Create final rag chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002CE86CE4810>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't

In [23]:
response=rag_chain.invoke({
    "input":"What is deep Learning"
})

In [24]:
response

{'input': 'What is deep Learning',
 'context': [Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision, RNNs and LSTMs for sequential data, and Transformers for\nnatural language processing. Training uses backpropagation and gradient descent\nto minimize a loss function.'),
  Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpayyjeji4\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision,

In [25]:
response["answer"]

'Deep learning is a subset of machine learning that uses neural networks with multiple hidden layers to analyze and interpret data.'

## Create RAG Chain Alternativee- Using LCEL

In [26]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel


In [27]:
custom_promt=ChatPromptTemplate.from_template("""You are an asisstant for question-answering tasks.
use the follwing pieceof retrieved context to answer the question
If you don't know the answer,just sat that you don't know.
Use three senetenes maximum and keep the answer concise.
Context:{context}
Question :{question}
                                
Answer:""")
custom_promt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't know the answer,just sat that you don't know.\nUse three senetenes maximum and keep the answer concise.\nContext:{context}\nQuestion :{question}\n\nAnswer:"), additional_kwargs={})])

In [28]:
## Format the output
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [29]:
### Build chain using LCEL
rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question":RunnablePassthrough()
    }
    |custom_promt
    |llm
    |StrOutputParser()
)
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002CE86CE4810>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an asisstant for question-answering tasks.\nuse the follwing pieceof retrieved context to answer the question\nIf you don't know the answer,just sat that you don't know.\nUse three senetenes maximum and keep the answer concise.\nContext:{context}\nQuestion :{question}\n\nAnswer:"), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2')
| StrOutputParser()

In [30]:
response=rag_chain_lcel.invoke('What is deep learning')
response

'Deep Learning is a subset of machine learning that uses neural networks with multiple hidden layers, consisting of input, hidden, and output layers connected by weighted edges. It employs various architectures such as CNNs for computer vision, RNNs and LSTMs for sequential data, and Transformers for natural language processing. Training uses backpropagation and gradient descent to minimize a loss function.'

In [66]:
retriever._get_relevant_documents(query='What is deep learning',run_manager="CallbackManagerForRetrieverRun")


[Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision, RNNs and LSTMs for sequential data, and Transformers for\nnatural language processing. Training uses backpropagation and gradient descent\nto minimize a loss function.'),
 Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpayyjeji4\\doc_1.txt'}, page_content='Deep Learning Fundamental\n\nDeep Learning is a subset of machine learning that uses neural networks with\nmultiple hidden layers. A neural network consists of input, hidden, and output\nlayers connected by weighted edges. Popular architectures include CNNs for\ncomputer vision, RNNs and LSTMs for sequential data, and Transfo

In [32]:
vectorstore

In [33]:
new_document="""
Reinforcement Learning

Reinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment. 
Instead of being told the correct answer (supervised learning), the agent learns through trial and error, receiving rewards or penalties for its actions.
einforcement learning is particularly useful when an agent must make a sequence of decisions where current 
actions influence future outcomes, and the optimal strategy is learned from experience rather than explicit examples.
"""

In [34]:
new_document

'\nReinforcement Learning\n\nReinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment. \nInstead of being told the correct answer (supervised learning), the agent learns through trial and error, receiving rewards or penalties for its actions.\neinforcement learning is particularly useful when an agent must make a sequence of decisions where current \nactions influence future outcomes, and the optimal strategy is learned from experience rather than explicit examples.\n'

In [35]:
new_doc=Document(
    page_content=new_document,
    metadata={"source":"manual_addition","topic":"reinforcement"}
)

In [36]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement'}, page_content='\nReinforcement Learning\n\nReinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment. \nInstead of being told the correct answer (supervised learning), the agent learns through trial and error, receiving rewards or penalties for its actions.\neinforcement learning is particularly useful when an agent must make a sequence of decisions where current \nactions influence future outcomes, and the optimal strategy is learned from experience rather than explicit examples.\n')

In [38]:
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement'}, page_content='Reinforcement Learning\n\nReinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment. \nInstead of being told the correct answer (supervised learning), the agent learns through trial and error, receiving rewards or penalties for its actions.\neinforcement learning is particularly useful when an agent must make a sequence of decisions where current \nactions influence future outcomes, and the optimal strategy is learned from experience rather'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement'}, page_content='strategy is learned from experience rather than explicit examples.')]

In [39]:
## Add newdoc to store
vectorstore.add_documents(new_chunks)

['6080edab-7cc9-4b15-a776-476639388334',
 'f4b3f922-10b9-4042-b933-50ca9b00e7d0']

In [41]:
print(vectorstore._collection.count())

22


In [44]:
result=rag_chain_lcel.invoke("What is reinforcement key icon")
result

'Reinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment.'

## Advanced RAG techniques

## Conversational memory

In [45]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage



In [50]:
## Create a prompt including chat history
contextualize_q_systemp_prompt="""
Given a chat history and the latest uer question which might reference content in 
chat history,formulate a standalone question 
which can be undestood without the chat history.DO Not answer the question ,just reformulate it if needed and otherwise
return it as it is"""
contextualize_q_systemp_prompt

'\nGiven a chat history and the latest uer question which might reference content in \nchat history,formulate a standalone question \nwhich can be undestood without the chat history.DO Not answer the question ,just reformulate it if needed and otherwise\nreturn it as it is'

In [51]:
contextualize_q_system_prompt=ChatPromptTemplate([
    ("system",contextualize_q_systemp_prompt),MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])

In [53]:
### Create history aware retriever
history_aware_retriever=create_history_aware_retriever(
llm,retriever,contextualize_q_system_prompt)

In [54]:
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002CE86CE4810>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag

In [59]:
qa_si_prompt="""You are an assistant for quustion-answering tasks.
Use the following pieces of retrievd ccontext to answer the question.
If you don't know the answer,just say that you dont know.
Use three sentences maximum and keep the answer concise.

Context:{context}"""

qa_prompt=ChatPromptTemplate.from_messages([
    ("system",qa_si_prompt),MessagesPlaceholder("chat_history"),
    ("human","{input}")
])
question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
conversation_rag_chain=create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational rag chain created")

Conversational rag chain created


In [60]:
chat_history=[]
result=conversation_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"What is machine learning?"
})
print("Q: What is Machine Learning")
print(f"A: {result['answer']}")

Q: What is Machine Learning
A: Machine Learning (ML) is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed. It allows machines to improve their performance on a task over time based on the data they receive. This type of learning enables computers to make predictions, classify data, and make decisions without human intervention.


In [61]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result['answer'])
])

In [64]:
result1=conversation_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"What is its main uses"
})
print("Q: What is it main uses?")
print(f"A: {result1['answer']}")

Q: What is it main uses?
A: Machine Learning (ML) has various applications across industries, including:

1. Predictive Analytics
2. Image Recognition
3. Natural Language Processing (NLP)
4. Recommendation Systems
5. Autonomous Vehicles


In [63]:
result1

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine Learning (ML) is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed. It allows machines to improve their performance on a task over time based on the data they receive. This type of learning enables computers to make predictions, classify data, and make decisions without human intervention.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What is its main uses',
 'context': [Document(metadata={'source': 'C:\\Users\\kanha\\AppData\\Local\\Temp\\tmpbi0kylce\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\nMachine Learning (ML) is a branch of artificial intelligence that enables\ncomputers to learn patterns from data without being explicitly programmed.\nThe main types of machine learning are supervised learning, unsuper

## Use GROQ LLM's

In [ ]:

llm



ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2')

In [ ]:
API_KEY=""
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [70]:
llm=ChatGroq(model="gemma2-9b-it",api_key=API_KEY)

In [71]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Gemma 2 9B', 'status': 'deprecated', 'release_date': '2024-06-27', 'last_updated': '2024-06-27', 'open_weights': True, 'max_input_tokens': 8192, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002CEA8123710>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002CE966CC050>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)